In [6]:
import pandas as pd
from pathlib import Path
from datetime import datetime
import sys
from pathlib import Path

In [7]:
def normalize_nsymbols(df):
    """Add a normalized NSymbols column where higher is better."""
    min_nsymbols = df['NSymbols'].min()
    df['NSymbols_norm'] = min_nsymbols / df['NSymbols']
    return df


def compute_mixed_score(df, weight_coverage=0.5):
    """
    Compute a mixed score combining Coverage and normalized NSymbols.
    weight_coverage: weight for coverage (0 to 1), the rest goes to NSymbols_norm
    """
    if 'NSymbols_norm' not in df.columns:
        df = normalize_nsymbols(df)
    
    df['Mixed_score'] = weight_coverage * df['Coverage'] + (1 - weight_coverage) * df['NSymbols_norm']
    return df


def rank_dcs(df, sort_by='Coverage', ascending=False):
    """
    Rank DCs by sorting on the specified column.
    
    Args:
        df: DataFrame with DC results
        sort_by: Column name to sort by
        ascending: Sort order (False = highest first, which is default for better scores)
    
    Returns:
        Sorted DataFrame
    """
    return df.sort_values(sort_by, ascending=ascending)


def parse_dc_to_predicates(dc_string):
    """
    Parse a DC string into a list of predicate strings.
    Works with DC Discovery format: "not(t1.x==t2.x and t1.y>t2.y)"
    
    Returns:
        List of predicate strings (keeps t1/t2 notation as-is)
    """
    import re
    
    # Remove "not(" prefix and trailing ")"
    dc_string = dc_string.strip()
    if dc_string.startswith('not('):
        dc_string = dc_string[4:]
    if dc_string.endswith(')'):
        dc_string = dc_string[:-1]
    
    # Split by " and " to get individual predicates
    pred_strings = dc_string.split(' and ')
    
    predicates = []
    for pred_str in pred_strings:
        pred_str = pred_str.strip()
        
        # Convert DC Discovery operators to SQL operators
        pred_str = pred_str.replace('==', '=')
        pred_str = pred_str.replace('<>', '!=')
        
        # Ensure spaces around operators for consistent format
        # Order matters - check longer operators first
        for op in ['<=', '>=', '!=', '=', '<', '>']:
            if op in pred_str:
                # Only add spaces if not already present
                pred_str = re.sub(f'([^\\s]){re.escape(op)}([^\\s])', f'\\1 {op} \\2', pred_str)
                break
        
        predicates.append(pred_str)
    
    return predicates

In [8]:
# Add workspace root to Python path
workspace_root = Path('/Users/pena/nl2c_projetcts/norma')
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from src.data import DatasetLoader

In [9]:
dataset_names = ['tax']

In [10]:
from src.evaluation.mapping import MappingEvaluator

for dataset_name in dataset_names:
    print(f"\n{'='*80}")
    print(f"Dataset: {dataset_name}")
    print(f"{'='*80}\n")
    
    # Load dataset using DatasetLoader
    data_path = f'/Users/pena/nl2c_projetcts/norma/data/{dataset_name}/'
    loader = DatasetLoader(data_path)
    df, sentences, sentence2dcs_map, predicate_space = loader.load()
    
    print(f"Loaded dataset: {dataset_name}")
    print(f"  Total sentences: {len(sentences)}")
    print(f"  Dataframe shape: {df.shape}")
    print(f"  Predicates: {len(predicate_space['predicates'])}")
    
    # Extract gold rules from sentence2dcs_map
    gold_rules = [rules for rules in sentence2dcs_map.values() if rules]
    print(f"  Ground truth constraints: {len(gold_rules)}")
    
    # Setup results tracking
    results_list = []
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    dc_profiling_path = Path(f'/Users/pena/nl2c_projetcts/baselines/profiling/results_dc_discovery/')
    
    # Test different top-k values
    top_ks = [5]
    
    # ============================================================================
    # EXACT DC DISCOVERY EVALUATION
    # ============================================================================
    csv_files = sorted(dc_profiling_path.glob(f'*{dataset_name}*exact*.csv'))
    if csv_files:
        exact_csv = csv_files[0]
        print(f"\n{'='*80}")
        print(f"EXACT DC DISCOVERY - {exact_csv.name}")
        print(f"{'='*80}")

        exact_dcs = pd.read_csv(exact_csv)
        exact_dcs = normalize_nsymbols(exact_dcs)
        exact_dcs = compute_mixed_score(exact_dcs, weight_coverage=0.5)
        print(f"Total DCs available: {len(exact_dcs)}")
        print(f"Testing top-k values: {top_ks}\n")

        for idx, top_k in enumerate(top_ks, 1):
            print(f"[{idx}/{len(top_ks)}] Evaluating Top-{top_k}...", end=' ', flush=True)
            
            ranked_df = rank_dcs(exact_dcs.copy(), sort_by="Mixed_score", ascending=False)
            top_dcs = ranked_df.head(top_k)
            converted_dcs = [parse_dc_to_predicates(dc) for dc in top_dcs['DC']]
            converted_dcs = [dc for dc in converted_dcs if dc]
            
            evaluator = MappingEvaluator(ground_truth=sentence2dcs_map, df=df)
            metrics = evaluator.evaluate_document_level(
                predicted_rules=converted_dcs,
                gold_rules=gold_rules
            )
            evaluator.close()
            
            print(f"✓")
            print(f"    Parsed: {len(converted_dcs)}/{top_k} DCs")
            print(f"    Constraint - P: {metrics['constraint_exact_precision']:.3f}, R: {metrics['constraint_exact_recall']:.3f}, F1: {metrics['constraint_exact_f1']:.3f}")
            print(f"    Violation  - P: {metrics['aggregate_violation_precision']:.3f}, R: {metrics['aggregate_violation_recall']:.3f}, F1: {metrics['aggregate_violation_f1']:.3f}")
            
            # Save results
            results_list.append({
                'dataset': dataset_name,
                'algorithm': 'dc_discovery',
                'mode': 'exact',
                'threshold': None,
                'top_k': top_k,
                'num_parsed': len(converted_dcs),
                'constraint_exact_precision': metrics['constraint_exact_precision'],
                'constraint_exact_recall': metrics['constraint_exact_recall'],
                'constraint_exact_f1': metrics['constraint_exact_f1'],
                'constraint_exact_tp': metrics['constraint_exact_tp'],
                'constraint_exact_fp': metrics['constraint_exact_fp'],
                'constraint_exact_fn': metrics['constraint_exact_fn'],
                'aggregate_violation_precision': metrics['aggregate_violation_precision'],
                'aggregate_violation_recall': metrics['aggregate_violation_recall'],
                'aggregate_violation_f1': metrics['aggregate_violation_f1'],
                'num_pred_constraints': metrics['num_pred_constraints'],
                'num_gold_constraints': metrics['num_gold_constraints'],
                'total_pred_violations': metrics['total_pred_violations'],
                'total_gold_violations': metrics['total_gold_violations'],
                'total_overlap_violations': metrics['total_overlap_violations']
            })
    else:
        print(f"\n⚠ No EXACT CSV found for {dataset_name}")

    # ============================================================================
    # APPROXIMATE DC DISCOVERY EVALUATION
    # ============================================================================
    approx_levels = ['approx_0.01']
    
    for approx_idx, approx_level in enumerate(approx_levels, 1):
        csv_files = sorted(dc_profiling_path.glob(f'*{dataset_name}*{approx_level}*.csv'))
        if csv_files:
            approx_csv = csv_files[0]
            threshold = approx_level.split('_')[1]  # Extract threshold value
            
            print(f"\n{'='*80}")
            print(f"{approx_level.upper().replace('_', ' ')} DC DISCOVERY - {approx_csv.name}")
            print(f"{'='*80}")
            
            approx_dcs = pd.read_csv(approx_csv)
            approx_dcs = normalize_nsymbols(approx_dcs)
            approx_dcs = compute_mixed_score(approx_dcs, weight_coverage=0.5)
            print(f"Total DCs available: {len(approx_dcs)}")
            print(f"Testing top-k values: {top_ks}\n")
            
            for idx, top_k in enumerate(top_ks, 1):
                print(f"[{idx}/{len(top_ks)}] Evaluating Top-{top_k}...", end=' ', flush=True)
                
                ranked_approx_df = rank_dcs(approx_dcs.copy(), sort_by="Mixed_score", ascending=False)
                top_dcs = ranked_approx_df.head(top_k)
                converted_dcs = [parse_dc_to_predicates(dc) for dc in top_dcs['DC']]
                converted_dcs = [dc for dc in converted_dcs if dc]
                
                evaluator = MappingEvaluator(ground_truth=sentence2dcs_map, df=df)
                metrics = evaluator.evaluate_document_level(
                    predicted_rules=converted_dcs,
                    gold_rules=gold_rules
                )
                evaluator.close()
                
                print(f"✓")
                print(f"    Parsed: {len(converted_dcs)}/{top_k} DCs")
                print(f"    Constraint - P: {metrics['constraint_exact_precision']:.3f}, R: {metrics['constraint_exact_recall']:.3f}, F1: {metrics['constraint_exact_f1']:.3f}")
                print(f"    Violation  - P: {metrics['aggregate_violation_precision']:.3f}, R: {metrics['aggregate_violation_recall']:.3f}, F1: {metrics['aggregate_violation_f1']:.3f}")
                
                # Save results
                results_list.append({
                    'dataset': dataset_name,
                    'algorithm': 'dc_discovery',
                    'mode': 'approximate',
                    'threshold': float(threshold),
                    'top_k': top_k,
                    'num_parsed': len(converted_dcs),
                    'constraint_exact_precision': metrics['constraint_exact_precision'],
                    'constraint_exact_recall': metrics['constraint_exact_recall'],
                    'constraint_exact_f1': metrics['constraint_exact_f1'],
                    'constraint_exact_tp': metrics['constraint_exact_tp'],
                    'constraint_exact_fp': metrics['constraint_exact_fp'],
                    'constraint_exact_fn': metrics['constraint_exact_fn'],
                    'aggregate_violation_precision': metrics['aggregate_violation_precision'],
                    'aggregate_violation_recall': metrics['aggregate_violation_recall'],
                    'aggregate_violation_f1': metrics['aggregate_violation_f1'],
                    'num_pred_constraints': metrics['num_pred_constraints'],
                    'num_gold_constraints': metrics['num_gold_constraints'],
                    'total_pred_violations': metrics['total_pred_violations'],
                    'total_gold_violations': metrics['total_gold_violations'],
                    'total_overlap_violations': metrics['total_overlap_violations']
                })
        else:
            print(f"\n⚠ No {approx_level.upper()} CSV found for {dataset_name}")
    
    print(f"\n{'='*80}")
    print(f"COMPLETED: {dataset_name}")
    print(f"{'='*80}")
    
    # Save results to CSV
    if results_list:
        results_dir = workspace_root / 'results' / 'baselines' / 'dc_discovery'
        results_dir.mkdir(parents=True, exist_ok=True)
        
        results_df = pd.DataFrame(results_list)
        results_file = results_dir / f'dc_discovery_{dataset_name}_{timestamp}.csv'
        results_df.to_csv(results_file, index=False)
        
        print(f"\n✓ Results saved to: {results_file}")
        print(f"  Total rows: {len(results_df)}")
    else:
        print(f"\n⚠ No results to save")



Dataset: tax



/Users/pena/nl2c_projetcts/norma/src/data/loader.py:51: UserWarning: Predicate space file exists at /Users/pena/nl2c_projetcts/norma/data/tax/tax_predicates.json, overriding with auto-generated predicates
  warnings.warn(


Saved to /Users/pena/nl2c_projetcts/norma/data/tax/tax_predicates.json
Loaded dataset: tax
  Total sentences: 130
  Dataframe shape: (99906, 15)
  Predicates: 174
  Ground truth constraints: 6

EXACT DC DISCOVERY - tax_trimmed_exact_20260212_142121.csv
Total DCs available: 15231
Testing top-k values: [5]

[1/1] Evaluating Top-5... 
Evaluating 5 predicted vs 6 gold constraints
Computing exact match constraint metrics...
  TP: 1, FP: 4, FN: 5
Executing gold constraints...
  Total unique gold violations: 468
Executing predicted constraints...
  Total unique pred violations: 1884
✓
    Parsed: 5/5 DCs
    Constraint - P: 0.200, R: 0.167, F1: 0.182
    Violation  - P: 0.004, R: 0.017, F1: 0.007

APPROX 0.01 DC DISCOVERY - tax_trimmed_approx_0.01000_20260212_142140.csv
Total DCs available: 18158
Testing top-k values: [5]

[1/1] Evaluating Top-5... 
Evaluating 5 predicted vs 6 gold constraints
Computing exact match constraint metrics...
  TP: 0, FP: 5, FN: 6
Executing gold constraints...
  To